# L3: Word Embeddings - Vector Representations of Words

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Basic  
**Lesson:** 3 of 15

---

## 📚 Learning Objectives

By the end of this lesson, you will:
- Understand what word embeddings are and why they're important
- Learn about Word2Vec and GloVe algorithms
- Train your own word embeddings
- Explore semantic relationships in embedding spaces
- Visualize word vectors

---

## 🧠 Concept: What are Word Embeddings?

**Word Embeddings** are dense vector representations of words that capture semantic meaning.

### Why Embeddings?
- **One-hot encoding:** Sparse, no semantic information
  - "king" = [1, 0, 0, 0, ...] (vocab size dimensions)
  - "queen" = [0, 1, 0, 0, ...]
  - No relationship captured!

- **Embeddings:** Dense, semantic relationships
  - "king" = [0.2, 0.8, -0.3, ...] (e.g., 300 dimensions)
  - "queen" = [0.3, 0.7, -0.2, ...]
  - Similar vectors for similar meanings!

### Key Properties
1. **Semantic Similarity:** Similar words have similar vectors
2. **Analogies:** king - man + woman ≈ queen
3. **Dimensionality Reduction:** From vocab_size to ~300 dimensions

---

In [ ]:
# Step 1: Install and Import Libraries
# !pip install gensim numpy matplotlib scikit-learn

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from gensim.models import Word2Vec
from gensim.test.utils import common_texts
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

## 📖 Part 1: Understanding Word2Vec

**Word2Vec** has two architectures:
1. **CBOW (Continuous Bag of Words):** Predicts target word from context
2. **Skip-gram:** Predicts context words from target word

### Training Process
- Input: Large text corpus
- Output: Vector for each word
- Method: Neural network trained to predict word contexts

---

In [ ]:
# Step 2: Prepare Training Data
# Using sample sentences for demonstration

sentences = [
    ['king', 'is', 'a', 'strong', 'man'],
    ['queen', 'is', 'a', 'wise', 'woman'],
    ['boy', 'is', 'a', 'young', 'man'],
    ['girl', 'is', 'a', 'young', 'woman'],
    ['prince', 'is', 'a', 'young', 'king'],
    ['princess', 'is', 'a', 'young', 'queen'],
    ['man', 'and', 'woman', 'are', 'human'],
    ['king', 'and', 'queen', 'are', 'royal'],
    ['boy', 'will', 'become', 'a', 'man'],
    ['girl', 'will', 'become', 'a', 'woman']
]

print(f"📊 Training corpus: {len(sentences)} sentences")
print(f"Sample: {sentences[0]}")

In [ ]:
# Step 3: Train Word2Vec Model

# Parameters:
# vector_size: Dimensionality of word vectors (e.g., 100, 300)
# window: Context window size (how many words to look around)
# min_count: Ignore words with frequency less than this
# sg: 0 for CBOW, 1 for Skip-gram

model = Word2Vec(
    sentences=sentences,
    vector_size=50,      # 50-dimensional vectors
    window=3,            # Context window of 3 words
    min_count=1,         # Include all words
    sg=1,                # Use Skip-gram
    epochs=100           # Training iterations
)

print("✅ Word2Vec model trained!")
print(f"Vocabulary size: {len(model.wv)}")
print(f"Vector dimensions: {model.wv.vector_size}")

In [ ]:
# Step 4: Explore Word Vectors

# Get vector for a word
king_vector = model.wv['king']
print(f"Vector for 'king': {king_vector[:10]}...")  # Show first 10 dimensions
print(f"Vector shape: {king_vector.shape}")

# Find similar words
print("\n🔍 Words similar to 'king':")
similar_words = model.wv.most_similar('king', topn=5)
for word, similarity in similar_words:
    print(f"  {word}: {similarity:.4f}")

## 🧮 Part 2: Word Analogies

One of the most fascinating properties of word embeddings is their ability to solve analogies:

**king - man + woman ≈ queen**

This works because:
- Vector arithmetic captures semantic relationships
- "king - man" removes the "male" component
- "+ woman" adds the "female" component
- Result is close to "queen"

---

In [ ]:
# Step 5: Test Word Analogies

def test_analogy(model, positive, negative, topn=3):
    """
    Test word analogy: positive[0] - negative[0] + positive[1] ≈ ?
    """
    try:
        result = model.wv.most_similar(
            positive=positive,
            negative=negative,
            topn=topn
        )
        return result
    except KeyError as e:
        return f"Word not in vocabulary: {e}"

# Test: king - man + woman = ?
print("🧪 Analogy: king - man + woman = ?")
result = test_analogy(model, positive=['king', 'woman'], negative=['man'])
for word, score in result:
    print(f"  {word}: {score:.4f}")

# Test: boy - man + woman = ?
print("\n🧪 Analogy: boy - man + woman = ?")
result = test_analogy(model, positive=['boy', 'woman'], negative=['man'])
for word, score in result:
    print(f"  {word}: {score:.4f}")

In [ ]:
# Step 6: Calculate Similarity Between Words

def calculate_similarity(model, word1, word2):
    """Calculate cosine similarity between two words"""
    try:
        similarity = model.wv.similarity(word1, word2)
        return similarity
    except KeyError as e:
        return f"Word not found: {e}"

# Test similarities
word_pairs = [
    ('king', 'queen'),
    ('king', 'man'),
    ('man', 'woman'),
    ('boy', 'girl'),
    ('king', 'girl')
]

print("📊 Word Similarities:")
for w1, w2 in word_pairs:
    sim = calculate_similarity(model, w1, w2)
    print(f"  {w1} <-> {w2}: {sim:.4f}")

## 📊 Part 3: Visualizing Embeddings

Since word vectors are high-dimensional (50-300 dimensions), we use **dimensionality reduction** to visualize them in 2D:

- **PCA (Principal Component Analysis):** Linear reduction
- **t-SNE:** Non-linear, preserves local structure

---

In [ ]:
# Step 7: Visualize Word Embeddings with PCA

def plot_embeddings(model, words=None):
    """
    Visualize word embeddings in 2D using PCA
    """
    if words is None:
        words = list(model.wv.index_to_key)  # All words
    
    # Get vectors for selected words
    vectors = np.array([model.wv[word] for word in words])
    
    # Reduce to 2D using PCA
    pca = PCA(n_components=2)
    vectors_2d = pca.fit_transform(vectors)
    
    # Plot
    plt.figure(figsize=(12, 8))
    plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], alpha=0.6, s=100)
    
    # Add labels
    for i, word in enumerate(words):
        plt.annotate(word, xy=(vectors_2d[i, 0], vectors_2d[i, 1]),
                    fontsize=12, alpha=0.8)
    
    plt.title('Word Embeddings Visualization (PCA)', fontsize=16)
    plt.xlabel('First Principal Component')
    plt.ylabel('Second Principal Component')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"✅ Visualized {len(words)} words")
    print(f"Explained variance: {pca.explained_variance_ratio_}")

# Visualize all words
plot_embeddings(model)

## 🎯 Part 4: Using Pre-trained Embeddings

Training embeddings requires large datasets. In practice, we often use **pre-trained embeddings**:

- **Word2Vec:** Trained on Google News (3 billion words)
- **GloVe:** Trained on Wikipedia + Gigaword (6 billion tokens)
- **FastText:** Handles out-of-vocabulary words

---

In [ ]:
# Step 8: Load Pre-trained Word2Vec (Google News)
# Note: This downloads ~1.5GB file - uncomment to use

# import gensim.downloader as api
# 
# print("📥 Downloading pre-trained Word2Vec model...")
# pretrained_model = api.load('word2vec-google-news-300')
# print("✅ Model loaded!")
# 
# # Test with pre-trained model
# print("\n🔍 Similar to 'computer':")
# for word, score in pretrained_model.most_similar('computer', topn=5):
#     print(f"  {word}: {score:.4f}")
# 
# # Test analogy
# print("\n🧪 Paris - France + Germany = ?")
# result = pretrained_model.most_similar(
#     positive=['Paris', 'Germany'],
#     negative=['France'],
#     topn=3
# )
# for word, score in result:
#     print(f"  {word}: {score:.4f}")

print("💡 Uncomment the code above to use pre-trained embeddings")

## 🎓 Key Takeaways

1. **Word embeddings** convert words to dense vectors that capture semantic meaning
2. **Word2Vec** learns embeddings by predicting word contexts
3. **Vector arithmetic** enables solving analogies (king - man + woman ≈ queen)
4. **Cosine similarity** measures how similar two words are
5. **Pre-trained embeddings** save time and work better with large vocabularies
6. **Visualization** helps understand relationships in embedding space

---

## 🔬 Exercises

1. Train Word2Vec on a larger corpus (use `common_texts` from gensim)
2. Compare CBOW vs Skip-gram performance
3. Test different vector sizes (50, 100, 300)
4. Create your own word analogies
5. Visualize embeddings with t-SNE instead of PCA

---

## 📚 Additional Resources

- **Paper:** "Efficient Estimation of Word Representations in Vector Space" (Mikolov et al., 2013)
- **Paper:** "GloVe: Global Vectors for Word Representation" (Pennington et al., 2014)
- **Tutorial:** [Gensim Word2Vec Tutorial](https://radimrehurek.com/gensim/models/word2vec.html)
- **Visualization:** [Embedding Projector](https://projector.tensorflow.org/)

---

## ➡️ Next Lesson

**L4: Attention Mechanisms** - Learn how transformers focus on relevant parts of input

---

**Happy Learning! 🚀**